# 🧠 GhostFaceNet / ArcFace `.h5` → `.tflite` Conversion Pipeline
**For Android AMS (Attendance Management System) — Offline Deployment**

---

## 📋 What This Notebook Does
Converts a pretrained **GhostFaceNet** or **ArcFace** `.h5` Keras model into four TFLite variants:

| Variant | Size | Speed | Accuracy | Use Case |
|---|---|---|---|---|
| `float32` | ~20MB | Baseline | 🟢 Best | High-end Android |
| `float16` | ~10MB | 1.5x faster | 🟢 Near-identical | Mid-range Android |
| `dynamic_int8` | ~5MB | 2–4x faster | 🟡 Very good | Low-end Android |
| `full_int8` | ~5MB | 2–4x faster + NNAPI | 🟡 Good | Old/low-end Android |

## ⚠️ Critical Research Findings (Read Before Running)
- GhostFaceNet uses `keras_cv_attention_models` backbone with **Ghost modules** — requires `model_surgery` before TFLite conversion
- `Conv2D with groups > 1` is **NOT natively supported** in TFLite — must be converted to split convolutions
- `GELU activation` must be converted to **approximate GELU** for TFLite compatibility
- Input normalization: `(pixel - 127.5) * 0.0078125` → maps `[0,255]` to `[-1, 1]`
- Input shape: `112 × 112 × 3` float32
- Output: `512-dimensional` embedding vector
- **Recommended TF version: 2.10–2.13** for best GhostFaceNet ↔ TFLite compat


---
## 📦 Step 1 — Install Dependencies

In [ ]:
# ─── Install all required packages ───────────────────────────────────────────
# TF 2.12 is the sweet spot: supports GhostFaceNet ops + modern TFLite converter
!pip install tensorflow==2.12.0
!pip install keras==2.12.0
!pip install keras-cv-attention-models  # GhostFaceNet backbone + model_surgery
!pip install numpy>=1.23.0
!pip install Pillow>=9.0
!pip install matplotlib>=3.5
!pip install tqdm
!pip install scikit-learn  # for cosine similarity verification

print("\n✅ All packages installed")

---
## 🔧 Step 2 — Imports & Version Check

In [ ]:
import os
import sys
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras

# Model surgery — required for GhostFaceNet → TFLite conversion
# This fixes Conv2D groups, GELU, and extract_patches incompatibilities
from keras_cv_attention_models import model_surgery

# ─── Version Check ────────────────────────────────────────────────────────────
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n🚀 GPU available: {len(gpus)} device(s)")
    for g in gpus:
        print(f"   → {g.name}")
else:
    print("\n⚠️  No GPU — running on CPU (conversion still works fine)")

---
## 📂 Step 3 — Configuration
**Set your model path and output directory here.**

In [ ]:
# ─── CONFIGURATION — Edit these paths ────────────────────────────────────────

# Path to your .h5 model file
# Examples:
#   GhostFaceNet:  'checkpoints/GhostFaceNet_W1.3_S1_ArcFace.h5'
#   GhostFaceNetV2:'checkpoints/GhostFaceNetV2_W1.3_S2_ArcFace.h5'
#   ArcFace R50:   'checkpoints/arcface_r50_v1.h5'
MODEL_H5_PATH = 'GhostFaceNet_W1.3_S1_ArcFace.h5'  # ← CHANGE THIS

# Output directory for .tflite files
OUTPUT_DIR = './tflite_output'

# Model type — affects how custom layers are handled
# Options: 'ghostfacenet_v1', 'ghostfacenet_v2', 'arcface_resnet', 'generic'
MODEL_TYPE = 'ghostfacenet_v1'  # ← CHANGE THIS

# Input shape (GhostFaceNet and most ArcFace models use 112x112)
INPUT_H, INPUT_W = 112, 112
INPUT_CHANNELS = 3

# Embedding dimension (GhostFaceNet = 512, MobileFaceNet = 192)
EMB_DIM = 512

# Path to a small calibration image folder (for INT8 full quantization)
# Can be any folder with face images — 50-200 images is enough
CALIB_IMAGE_DIR = './calibration_images'  # ← Optional but recommended for INT8
CALIB_IMAGE_COUNT = 100  # How many images to use for calibration

# ─── Create output dir ────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"📁 Model path      : {MODEL_H5_PATH}")
print(f"🔷 Model type      : {MODEL_TYPE}")
print(f"📐 Input shape     : ({INPUT_H}, {INPUT_W}, {INPUT_CHANNELS})")
print(f"📊 Embedding dim   : {EMB_DIM}")

---
## 🔄 Step 4 — Load the `.h5` Model

In [ ]:
# ─── Load Keras H5 Model ──────────────────────────────────────────────────────
# compile=False is important — we don't need the training graph,
# only the inference graph for TFLite conversion

print(f"🔃 Loading model from: {MODEL_H5_PATH}")

if not os.path.exists(MODEL_H5_PATH):
    raise FileNotFoundError(
        f"\n❌ Model file not found: {MODEL_H5_PATH}\n"
        f"Download GhostFaceNet pretrained weights from:\n"
        f"   https://github.com/HamadYA/GhostFaceNets\n"
        f"   (Google Drive links in the README)\n"
        f"Or DeepFace GhostFaceNet weights from:\n"
        f"   https://huggingface.co/serengil/deepface_models"
    )

model = tf.keras.models.load_model(MODEL_H5_PATH, compile=False)

# Print model summary
print(f"\n✅ Model loaded successfully")
print(f"   Input  shape : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")
print(f"   Total params : {model.count_params():,}")

# Verify input/output dimensions
assert len(model.input_shape) == 4, "Expected 4D input: (batch, H, W, C)"
_, h, w, c = model.input_shape
print(f"\n🔍 Input: {h}×{w}×{c} | Output: {model.output_shape[-1]}D embedding")

---
## 🔬 Step 5 — Model Surgery for TFLite Compatibility

**Why this is required for GhostFaceNet:**
- GhostFaceNet V2 uses **DFC attention** which has `tf.image.extract_patches` — not supported natively in TFLite
- Ghost modules use `Conv2D with groups > 1` — TFLite kernel only supports `groups=1`
- `GELU activation` must be converted to approximate GELU
- `model_surgery.prepare_for_tflite()` handles ALL of these in one call

In [ ]:
# ─── Model Surgery ────────────────────────────────────────────────────────────
# Required for GhostFaceNet models built with keras_cv_attention_models
# Safe to run on ArcFace ResNet models too — it's a no-op if not needed

def apply_model_surgery(model, model_type='ghostfacenet_v1'):
    """
    Apply TFLite compatibility fixes based on model type.
    
    GhostFaceNetV1: Needs groups conv fix only
    GhostFaceNetV2: Needs groups conv + extract_patches + GELU fix
    ArcFace/Generic: May not need surgery, but apply defensively
    """
    print(f"🔧 Applying model surgery for: {model_type}")
    
    try:
        if model_type in ('ghostfacenet_v1', 'ghostfacenet_v2', 'generic'):
            # prepare_for_tflite() combines:
            # 1. convert_groups_conv2d_2_split_conv2d  → fixes grouped convolutions
            # 2. convert_gelu_to_approximate           → fixes GELU activation  
            # 3. convert_extract_patches_to_conv       → fixes tf.image.extract_patches
            model = model_surgery.prepare_for_tflite(model)
            print("   ✅ Groups Conv2D → SplitConv2D")
            print("   ✅ GELU → GELU approximate")
            print("   ✅ extract_patches → Conv2D equivalent")
        elif model_type == 'arcface_resnet':
            # ResNet-based ArcFace is usually TFLite-compatible without surgery
            # But apply groups fix defensively
            try:
                model = model_surgery.convert_groups_conv2d_2_split_conv2d(model)
                print("   ✅ Groups Conv2D fix applied (defensive)")
            except Exception:
                print("   ℹ️  No grouped convolutions found — skipping")
        
        print(f"\n✅ Model surgery complete")
        print(f"   Output shape after surgery: {model.output_shape}")
        return model
    
    except Exception as e:
        print(f"\n⚠️  model_surgery failed: {e}")
        print("   Falling back to SELECT_TF_OPS conversion (larger but safe)")
        return model  # Return original — will use SELECT_TF_OPS fallback

model_for_conversion = apply_model_surgery(model, MODEL_TYPE)

# Quick sanity check — run a forward pass to confirm surgery didn't break anything
dummy_input = np.random.uniform(0, 1, (1, INPUT_H, INPUT_W, INPUT_CHANNELS)).astype(np.float32)
original_output = model.predict(dummy_input, verbose=0)
surgery_output = model_for_conversion.predict(dummy_input, verbose=0)

max_diff = np.max(np.abs(original_output - surgery_output))
print(f"\n🔬 Sanity check — max output diff after surgery: {max_diff:.8f}")
if max_diff < 1e-4:
    print("   ✅ Surgery preserved model accuracy (diff < 1e-4)")
else:
    print(f"   ⚠️  Larger diff detected ({max_diff:.6f}) — review surgery output")

---
## 🔄 Step 6 — Conversion Helper Functions

In [ ]:
# ─── Preprocessing helper ─────────────────────────────────────────────────────
def preprocess_image(img_path_or_array, target_size=(112, 112)):
    """
    GhostFaceNet / ArcFace normalization:
    [0, 255] RGB → [-1.0, 1.0] float32
    Formula: (pixel - 127.5) * 0.0078125
    """
    if isinstance(img_path_or_array, str):
        img = Image.open(img_path_or_array).convert('RGB')
        img = img.resize(target_size)
        img = np.array(img, dtype=np.float32)
    else:
        img = img_path_or_array.astype(np.float32)
    
    # Normalize to [-1, 1]
    img = (img - 127.5) * 0.0078125
    return np.expand_dims(img, axis=0)  # (1, H, W, 3)


# ─── Representative dataset for INT8 calibration ─────────────────────────────
def make_representative_dataset(image_dir, count=100, target_size=(112, 112)):
    """
    Creates a generator that yields preprocessed face images.
    Used for full INT8 quantization calibration.
    The converter uses these to compute activation ranges.
    """
    image_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        image_paths.extend(glob.glob(os.path.join(image_dir, '**', ext), recursive=True))
    
    if not image_paths:
        print(f"⚠️  No images found in {image_dir} — using random noise for calibration")
        def noise_generator():
            for _ in range(count):
                # Random noise in [-1, 1] range (matches face normalization)
                yield [np.random.uniform(-1, 1, (1, *target_size, 3)).astype(np.float32)]
        return noise_generator
    
    random.shuffle(image_paths)
    selected = image_paths[:min(count, len(image_paths))]
    print(f"📸 Using {len(selected)} images for INT8 calibration from {image_dir}")
    
    def generator():
        for path in selected:
            try:
                yield [preprocess_image(path, target_size)]
            except Exception:
                pass  # Skip corrupted images
    
    return generator


# ─── TFLite inference helper ──────────────────────────────────────────────────
def run_tflite_inference(tflite_path, input_array):
    """
    Run inference with a .tflite model.
    Handles both float32 and int8 quantized models automatically.
    """
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Handle quantized input (INT8 models need dequantized input)
    input_dtype = input_details[0]['dtype']
    if input_dtype == np.int8:
        scale, zero_point = input_details[0]['quantization']
        input_data = (input_array / scale + zero_point).astype(np.int8)
    elif input_dtype == np.uint8:
        scale, zero_point = input_details[0]['quantization']
        input_data = (input_array / scale + zero_point).astype(np.uint8)
    else:
        input_data = input_array.astype(np.float32)
    
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    
    # Dequantize output if needed
    output_dtype = output_details[0]['dtype']
    if output_dtype in (np.int8, np.uint8):
        scale, zero_point = output_details[0]['quantization']
        output = (output.astype(np.float32) - zero_point) * scale
    
    return output


# ─── Cosine similarity ────────────────────────────────────────────────────────
def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a = a.flatten()
    b = b.flatten()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


print("✅ Helper functions defined")

---
## 🟢 Step 7 — Convert to Float32 TFLite (Baseline)
No quantization — maximum accuracy, largest file size.

In [ ]:
# ─── Float32 Conversion ───────────────────────────────────────────────────────
print("\n🔄 Converting to Float32 TFLite...")

output_path_f32 = os.path.join(OUTPUT_DIR, 'ghostfacenet_float32.tflite')

converter_f32 = tf.lite.TFLiteConverter.from_keras_model(model_for_conversion)

# Allow select TF ops as fallback for any remaining unsupported ops
converter_f32.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS  # Fallback — enables all TF ops
]
converter_f32._experimental_lower_tensor_list_ops = False

tflite_f32 = converter_f32.convert()

with open(output_path_f32, 'wb') as f:
    f.write(tflite_f32)

size_f32 = os.path.getsize(output_path_f32) / (1024 * 1024)
print(f"\n✅ Float32 TFLite saved → {output_path_f32}")
print(f"   Size: {size_f32:.2f} MB")

---
## 🔵 Step 8 — Convert to Float16 TFLite
Weights compressed to 16-bit. ~2× smaller, minimal accuracy loss.

In [ ]:
# ─── Float16 Quantization ─────────────────────────────────────────────────────
print("\n🔄 Converting to Float16 TFLite...")

output_path_f16 = os.path.join(OUTPUT_DIR, 'ghostfacenet_float16.tflite')

converter_f16 = tf.lite.TFLiteConverter.from_keras_model(model_for_conversion)
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]
converter_f16.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter_f16._experimental_lower_tensor_list_ops = False

tflite_f16 = converter_f16.convert()

with open(output_path_f16, 'wb') as f:
    f.write(tflite_f16)

size_f16 = os.path.getsize(output_path_f16) / (1024 * 1024)
print(f"\n✅ Float16 TFLite saved → {output_path_f16}")
print(f"   Size: {size_f16:.2f} MB")
print(f"   Compression vs Float32: {size_f32 / size_f16:.1f}×")

---
## 🟡 Step 9 — Convert to Dynamic Range INT8 TFLite
Weights quantized to INT8. ~4× smaller, runtime dequantization. No calibration data needed.

In [ ]:
# ─── Dynamic Range INT8 Quantization ─────────────────────────────────────────
# Weights are statically quantized to INT8
# Activations are dynamically quantized at runtime
# No representative dataset needed — best balance for old Android devices
print("\n🔄 Converting to Dynamic INT8 TFLite...")

output_path_dyn = os.path.join(OUTPUT_DIR, 'ghostfacenet_dynamic_int8.tflite')

converter_dyn = tf.lite.TFLiteConverter.from_keras_model(model_for_conversion)
converter_dyn.optimizations = [tf.lite.Optimize.DEFAULT]
# Note: DEFAULT without representative_dataset = dynamic range quantization
converter_dyn.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter_dyn._experimental_lower_tensor_list_ops = False

tflite_dyn = converter_dyn.convert()

with open(output_path_dyn, 'wb') as f:
    f.write(tflite_dyn)

size_dyn = os.path.getsize(output_path_dyn) / (1024 * 1024)
print(f"\n✅ Dynamic INT8 TFLite saved → {output_path_dyn}")
print(f"   Size: {size_dyn:.2f} MB")
print(f"   Compression vs Float32: {size_f32 / size_dyn:.1f}×")

---
## 🔴 Step 10 — Convert to Full INT8 TFLite (with Calibration)
Both weights AND activations quantized to INT8. Requires calibration images. Enables **Android NNAPI** hardware acceleration on old devices.

In [ ]:
# ─── Full INT8 Quantization (with representative dataset) ─────────────────────
# This is the best option for old/low-end Android phones:
#   - Enables NNAPI hardware acceleration on Snapdragon, MediaTek, etc.
#   - Smallest model size + fastest inference
#   - Requires ~100 calibration images to compute activation ranges
print("\n🔄 Converting to Full INT8 TFLite...")
print(f"   Using calibration dir: {CALIB_IMAGE_DIR}")

output_path_int8 = os.path.join(OUTPUT_DIR, 'ghostfacenet_full_int8.tflite')

# Create representative dataset generator
rep_dataset = make_representative_dataset(
    CALIB_IMAGE_DIR,
    count=CALIB_IMAGE_COUNT,
    target_size=(INPUT_H, INPUT_W)
)

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model_for_conversion)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = rep_dataset
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
# Keep input/output as float32 for easier integration in React Native
# (The model internally uses INT8 but accepts/returns float32)
converter_int8.inference_input_type = tf.float32
converter_int8.inference_output_type = tf.float32
converter_int8._experimental_lower_tensor_list_ops = False

try:
    tflite_int8 = converter_int8.convert()
    with open(output_path_int8, 'wb') as f:
        f.write(tflite_int8)
    size_int8 = os.path.getsize(output_path_int8) / (1024 * 1024)
    print(f"\n✅ Full INT8 TFLite saved → {output_path_int8}")
    print(f"   Size: {size_int8:.2f} MB")
    print(f"   Compression vs Float32: {size_f32 / size_int8:.1f}×")
except Exception as e:
    print(f"\n⚠️  Full INT8 conversion failed: {e}")
    print("   Falling back to Dynamic INT8 — this is still good for old devices")
    import shutil
    shutil.copy(output_path_dyn, output_path_int8)
    size_int8 = size_dyn

---
## ✅ Step 11 — Verify All Models (Accuracy Check)
Run the same face through all 4 models and compare cosine similarity between outputs.

In [ ]:
# ─── Accuracy Verification ────────────────────────────────────────────────────
# Run identical input through Keras + all 4 TFLite models
# Cosine similarity to original Keras output should be > 0.999 for float,
# > 0.99 for INT8

print("\n🔬 Running accuracy verification...")
print("   Comparing TFLite outputs to original Keras model output\n")

# Generate a consistent test input (simulated face)
np.random.seed(42)
test_input = np.random.uniform(0, 255, (1, INPUT_H, INPUT_W, INPUT_CHANNELS)).astype(np.float32)
test_input_norm = (test_input - 127.5) * 0.0078125  # Apply GhostFaceNet normalization

# Ground truth from original Keras model
keras_embedding = model.predict(test_input_norm, verbose=0)
print(f"Keras embedding shape: {keras_embedding.shape}")
print(f"Keras embedding norm : {np.linalg.norm(keras_embedding):.4f}")

# Test each TFLite model
results = []
models_to_test = [
    ('float32',       output_path_f32),
    ('float16',       output_path_f16),
    ('dynamic_int8',  output_path_dyn),
    ('full_int8',     output_path_int8),
]

for name, path in models_to_test:
    if not os.path.exists(path):
        print(f"   ⚠️  {name}: file not found, skipping")
        continue
    try:
        tflite_embedding = run_tflite_inference(path, test_input_norm)
        sim = cosine_similarity(keras_embedding, tflite_embedding)
        l2_diff = np.linalg.norm(keras_embedding - tflite_embedding)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        
        status = "🟢" if sim > 0.999 else ("🟡" if sim > 0.99 else "🔴")
        print(f"   {status} {name:20s} | cosine_sim: {sim:.6f} | L2: {l2_diff:.6f} | size: {size_mb:.2f}MB")
        results.append({'name': name, 'sim': sim, 'l2': l2_diff, 'size': size_mb})
    except Exception as e:
        print(f"   ❌ {name:20s} | Error: {e}")

print("\n📊 Threshold guide: cosine_sim > 0.999 = excellent | > 0.99 = acceptable | < 0.99 = review needed")

---
## 📊 Step 12 — Visualize Results

In [ ]:
# ─── Visualization ────────────────────────────────────────────────────────────
if results:
    names = [r['name'] for r in results]
    sims = [r['sim'] for r in results]
    sizes = [r['size'] for r in results]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('GhostFaceNet TFLite Conversion Results', fontsize=14, fontweight='bold')

    # Cosine similarity chart
    colors = ['#10B981' if s > 0.999 else '#F59E0B' if s > 0.99 else '#EF4444' for s in sims]
    bars1 = axes[0].bar(names, sims, color=colors, edgecolor='white', linewidth=1.5)
    axes[0].set_title('Accuracy (Cosine Similarity to Keras)', fontweight='bold')
    axes[0].set_ylabel('Cosine Similarity')
    axes[0].set_ylim([min(sims) - 0.005, 1.001])
    axes[0].axhline(y=0.999, color='#10B981', linestyle='--', alpha=0.5, label='Excellent (0.999)')
    axes[0].axhline(y=0.990, color='#F59E0B', linestyle='--', alpha=0.5, label='Acceptable (0.990)')
    axes[0].legend(fontsize=8)
    for bar, val in zip(bars1, sims):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                     f'{val:.5f}', ha='center', va='bottom', fontsize=8)

    # Model size chart
    bars2 = axes[1].bar(names, sizes, color='#4F46E5', edgecolor='white', linewidth=1.5)
    axes[1].set_title('Model Size (MB)', fontweight='bold')
    axes[1].set_ylabel('Size (MB)')
    for bar, val in zip(bars2, sizes):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.1f}MB', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, 'conversion_results.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n📊 Chart saved → {chart_path}")

---
## 🔍 Step 13 — Inspect TFLite Model Details
Print input/output tensor specs — critical for React Native integration.

In [ ]:
# ─── TFLite Tensor Inspector ──────────────────────────────────────────────────
# Run this to get exact tensor names, shapes, and dtypes
# These values must match your react-native-fast-tflite configuration

def inspect_tflite(path, name):
    print(f"\n{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    
    print(f" INPUT  tensor:")
    print(f"   name  : {inp['name']}")
    print(f"   shape : {inp['shape'].tolist()} → (batch, H, W, C)")
    print(f"   dtype : {inp['dtype'].__name__}")
    if inp.get('quantization', (0, 0)) != (0, 0):
        print(f"   quant : scale={inp['quantization'][0]:.6f}, zero_point={inp['quantization'][1]}")
    
    print(f" OUTPUT tensor:")
    print(f"   name  : {out['name']}")
    print(f"   shape : {out['shape'].tolist()} → (batch, embedding_dim)")
    print(f"   dtype : {out['dtype'].__name__}")
    if out.get('quantization', (0, 0)) != (0, 0):
        print(f"   quant : scale={out['quantization'][0]:.6f}, zero_point={out['quantization'][1]}")

print("🔍 TFLite Model Tensor Details (for React Native integration)\n")
for name, path in models_to_test:
    if os.path.exists(path):
        inspect_tflite(path, name)

print(f"\n\n📌 React Native TFLite usage note:")
print(f"   Input  : float32 array of shape [1, {INPUT_H}, {INPUT_W}, 3]")
print(f"   Values : normalized to [-1, 1] via: (pixel - 127.5) * 0.0078125")
print(f"   Output : float32 array of shape [1, {EMB_DIM}]")
print(f"   Usage  : model.runSync([input]) in worklet")

---
## 📱 Step 14 — AMS Integration Test
Simulate enrollment + recognition as your AMS app would do it.

In [ ]:
# ─── AMS Enrollment + Recognition Simulation ─────────────────────────────────
# Picks the best available TFLite model and simulates:
#   1. Employee enrollment (5 embeddings → centroid)
#   2. Recognition (live embedding vs stored embeddings)
# This is exactly how your React Native worklet will use the model

print("\n🏢 AMS Integration Test — Simulating Enrollment + Recognition\n")

# Pick best model: prefer dynamic_int8 (best for old Android devices)
test_model_path = output_path_dyn if os.path.exists(output_path_dyn) else output_path_f32
print(f"Using model: {os.path.basename(test_model_path)}")

np.random.seed(0)

# ── Simulate enrollment for 3 employees ───────────────────────────────────────
def simulate_enrollment(model_path, n_employees=3, embeddings_per_employee=5):
    """Simulate capturing 5 face images per employee and computing centroid."""
    employees = {}
    for emp_id in range(n_employees):
        # Simulate 5 slightly different captures of the same face
        base_face = np.random.uniform(0, 255, (INPUT_H, INPUT_W, INPUT_CHANNELS)).astype(np.float32)
        embeddings = []
        for _ in range(embeddings_per_employee):
            # Add small noise to simulate natural variation between captures
            noisy_face = base_face + np.random.normal(0, 5, base_face.shape)
            noisy_face = np.clip(noisy_face, 0, 255)
            norm_face = (noisy_face - 127.5) * 0.0078125
            norm_face = np.expand_dims(norm_face, 0)  # (1, H, W, 3)
            emb = run_tflite_inference(model_path, norm_face).flatten()
            embeddings.append(emb)
        
        centroid = np.mean(embeddings, axis=0)  # Compute centroid embedding
        centroid /= np.linalg.norm(centroid)    # L2-normalize centroid
        
        employees[f'employee_{emp_id}'] = {
            'embeddings': embeddings,
            'centroid': centroid,
            'base_face': base_face
        }
        print(f"   Enrolled employee_{emp_id}: {len(embeddings)} embeddings captured")
    return employees

employees = simulate_enrollment(test_model_path)

# ── Simulate recognition ───────────────────────────────────────────────────────
print("\n🔍 Recognition test (max-sim voting against all 5 stored embeddings):")
THRESHOLD = 0.60

for target_id, emp_data in employees.items():
    # Simulate a new scan of the same employee (slightly different capture)
    live_face = emp_data['base_face'] + np.random.normal(0, 8, emp_data['base_face'].shape)
    live_face = np.clip(live_face, 0, 255)
    live_norm = (live_face - 127.5) * 0.0078125
    live_embedding = run_tflite_inference(test_model_path, np.expand_dims(live_norm, 0)).flatten()
    live_embedding /= np.linalg.norm(live_embedding)
    
    # Max-sim voting: compare against all stored embeddings
    all_scores = {}
    for emp_id, data in employees.items():
        scores = [cosine_similarity(live_embedding, e) for e in data['embeddings']]
        all_scores[emp_id] = max(scores)  # Take max score
    
    best_match = max(all_scores, key=all_scores.get)
    best_score = all_scores[best_match]
    
    correct = best_match == target_id and best_score >= THRESHOLD
    status = '✅' if correct else '❌'
    print(f"   {status} Scanning {target_id} → matched: {best_match} (score: {best_score:.4f}, threshold: {THRESHOLD})")

print("\n🎉 AMS integration test complete!")

---
## 📋 Step 15 — Final Summary & Recommendation

In [ ]:
# ─── Final Summary ────────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  ✅  CONVERSION COMPLETE — SUMMARY")
print("═"*60)

output_files = [
    ('Float32  (baseline)',  output_path_f32,  'High-end Android (API 28+)'),
    ('Float16  (half)',      output_path_f16,  'Mid-range Android (API 26+)'),
    ('Dyn INT8 (no calib)', output_path_dyn,  '✅ RECOMMENDED: Old Android (API 21+)'),
    ('Full INT8 (calib)',    output_path_int8, 'Best perf + NNAPI on old devices'),
]

for label, path, use_case in output_files:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  {label:25s} {size:6.2f} MB  →  {use_case}")
    else:
        print(f"  {label:25s} {'NOT CREATED':10s}")

print("═"*60)
print("\n📱 FOR YOUR AMS (react-native-fast-tflite):")
print(f"   Copy → android/app/src/main/assets/ghostfacenet.tflite")
print(f"   Use  → ghostfacenet_dynamic_int8.tflite (best old-device support)")
print()
print("🔧 In worklet (replace mobilefacenet references):")
print(f"   Input  : float32[1][{INPUT_H}][{INPUT_W}][3]")
print(f"   Norm   : (pixel - 127.5) * 0.0078125  →  [-1, 1]")
print(f"   Output : float32[1][{EMB_DIM}]  (512-dim embedding)")
print(f"   API    : model.runSync([input]) — same as before")
print()
print(f"📁 All output files in: {os.path.abspath(OUTPUT_DIR)}")
print("═"*60)

---
## 📝 Notes for React Native Integration

### Key differences vs MobileFaceNet you had before:

| Property | MobileFaceNet (old) | GhostFaceNet (new) |
|---|---|---|
| Input size | 192 × 192 | **112 × 112** |
| Embedding dim | 192 | **512** |
| Normalization | `[-1, 1]` | `[-1, 1]` (same formula) |
| Accuracy (LFW) | 99.55% | **99.78%** |
| Model size | ~4 MB | ~5 MB (INT8) |

### Update your `config.ts`:
```ts
export const FACE_MODEL_CONFIG = {
  inputSize: 112,         // Changed from 192
  embeddingDim: 512,      // Changed from 192
  normalize: (pixel: number) => (pixel - 127.5) * 0.0078125,  // Same
  modelFile: 'ghostfacenet.tflite',
};
```

### Update resize plugin call in worklet:
```ts
// Change targetWidth and targetHeight from 192 to 112
const resized = resize(frame, {
  scale: { width: 112, height: 112 },  // Was 192
  pixelFormat: 'rgb',
  dataType: 'float32',  // NEVER change this
});
```

### Cosine similarity threshold stays the same:
- Global threshold: `0.60`
- Per-employee cap: `0.72`
- 512-dim embeddings are more discriminative → you may be able to raise global threshold to `0.65`
